# Optuna로 딥러닝 하이퍼파라미터 자동 튜닝

# 1️⃣ 왜 Optuna가 필요한가?

딥러닝 모델 성능은 다음 요소에 크게 의존한다:

- Learning Rate
- Optimizer 종류
- Batch Size
- Dropout
- Weight Decay
- Scheduler
- Layer Freeze 여부

이걸 사람이 직접 조합하면?

```
lr: 3개
optimizer: 3개
batch: 3개
dropout: 3개
→ 3⁴ = 81가지 실험
```

➡️ 시간 낭비

➡️ GPU 자원 낭비

➡️ 체계적 탐색 불가

💡 그래서 사용하는 것이 **Hyperparameter Optimization (HPO)**

💡 그 중 가장 강력하고 실전적인 도구 → **Optuna**

---

# 2️⃣ Optuna란?

- Python 기반 자동 하이퍼파라미터 탐색 라이브러리
- Bayesian Optimization 기반
- Pruning(조기 종료) 지원
- PyTorch, TensorFlow 완벽 호환
- 병렬 GPU 튜닝 가능

공식 사이트: [https://optuna.org](https://optuna.org/)
# Optuna 기본 개념

## 핵심 구성요소

| 개념 | 설명 |
| --- | --- |
| Study | 실험 전체 관리 객체 |
| Trial | 한 번의 실험 |
| suggest | 하이퍼파라미터 샘플링 |
| objective | 최적화 대상 함수 |
| pruning | 성능 안좋은 실험 조기 종료 |

---

# 4️⃣ 딥러닝 예제 (PyTorch + Optuna)

## 🎯 CIFAR10 분류 예제

---

## 1. 라이브러리 설치

```python
pip install optuna
```

# Pruning(조기 종료)의 원리

일반 학습:

```
Trial 1 → 50 epoch
Trial 2 → 50 epoch
Trial 3 → 50 epoch
```

Optuna Pruning:

```
Trial 1 → 50 epoch
Trial 2 → 5 epoch (성능 낮음 → 종료)
Trial 3 → 8 epoch (성능 낮음 → 종료)
```

➡️ GPU 시간 60~80% 절약 가능

---

# 6️⃣ 결과 시각화

```python
import optuna.visualization as vis

vis.plot_optimization_history(study)
vis.plot_param_importances(study)
```

---

# 7️⃣ GPU 병렬 튜닝 방법

## 방법 1️⃣ n_jobs 사용 (CPU 병렬)

```python
study.optimize(objective,n_trials=50,n_jobs=4)
```


## 방법 2️⃣ RDB 기반 병렬 GPU 튜닝

```python
study=optuna.create_study(
study_name="cnn_study",
storage="sqlite:///example.db",
load_if_exists=True,
direction="maximize"
)
```

다른 터미널에서 동시에 실행하면 GPU 병렬 실험 가능

---

# 8️⃣ 실전에서 자주 튜닝하는 것들

종류	추천 탐색
lr	loguniform
weight_decay	loguniform
dropout	float
optimizer	categorical
scheduler	categorical
batch_size	categorical
layer freeze	boolean

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import optuna

device = "cuda" if torch.cuda.is_available() else "cpu"

# 데이터셋
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

val_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

def objective(trial):

    # 🔥 하이퍼파라미터 탐색 공간 설정
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-1) # 필수사용
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128]) # 필수사용
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"]) # Adam 사용으로 기본 잡음
    dropout_rate = trial.suggest_float("dropout", 0.2, 0.5) 

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False
    )

    # 간단한 CNN 모델 Dropout 을 사용하지 않을경우 objective 함수 밖으로 뺀다
    model = nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Dropout(dropout_rate),
        nn.Linear(32 * 16 * 16, 10)
    ).to(device)

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9) # momentum 0.7~0.9사이 설정

    criterion = nn.CrossEntropyLoss()

    # 학습
    for epoch in range(5):

        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total

        # 🔥 Pruning 조건
        trial.report(accuracy, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best Trial:")
print(study.best_trial.params)